In [ ]:
import requests
import time

def chat_with_model(url, messages):
    response = requests.post(
        url,
        json={"model": "deepseek-r1:1.5b", "messages": messages, "stream": False},
    )
    response_json = response.json()
    return response_json.get("message", {}).get("content", "")

def run_conversation():
    url1 = "http://localhost:11434/api/chat"
    url2 = "http://localhost:11435/api/chat"

    # Set initial system prompts for each model
    messages = [
        {"role": "system", "content": "You are a friendly AI that loves discussing ideas and asking follow-up questions."},
        {"role": "user", "content": "Hello! Let's have a fun and engaging conversation."}
    ]

    for _ in range(10):  # Run for 10 exchanges
        response1 = chat_with_model(url1, messages)
        print("Model 1:", response1)
        messages.append({"role": "assistant", "content": response1})

        time.sleep(1)

        response2 = chat_with_model(url2, messages)
        print("Model 2:", response2)
        messages.append({"role": "assistant", "content": response2})

        time.sleep(1)

if __name__ == "__main__":
    run_conversation()



In [ ]:
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns
import os

#conversationData = pd.read_csv('video-game-text-corpora/Torchlight II/data/dataset_200630.csv')
conversationData = pd.read_csv('video-game-text-corpora/Star Wars: Knights of the Old Republic/data/dataset_20200716.csv')

# print(conversationData.head()) # Display the first 5 rows of the dataset
# description = conversationData.describe()
# print(description)
# print(conversationData.shape) # Display the number of rows and columns in the dataset
# print(conversationData.columns) # Display the column names in the dataset
# print(conversationData.info()) # Display the information about the dataset
# print(conversationData.isnull().sum()) # Display the number of missing values in the dataset

df = pd.DataFrame(conversationData)

# Get the count of repeated data in the specific column
repeated_counts = df['speaker'].value_counts()
filtered_rows = df.loc[df['speaker'] == 'Bastila']
filtered_rows.head(100)

In [27]:
import re

# Read the JSON file
with open('/home/sma/deepseekPersona/Azrael_speech_data.json', 'r') as file:
    lines = file.readlines()

# Initialize a flag to alternate between [ and ]
replace_with_bracket = True

# Process each line
for i in range(len(lines)):
    # Check for empty line
    if lines[i].strip() == '':

            lines[i] = ']},{"conversations":[\n'


# Write the modified lines back to the file
with open('/home/sma/deepseekPersona/Azrael_speech_data.json', 'w') as file:
    file.writelines(lines)

In [ ]:
from unsloth import FastLanguageModel
import torch

# Define configurations for loading the model
max_seq_length = 2048 
dtype = None  # Automatically choose the best data type (float16, bfloat16, etc.) 
load_in_4bit = True  # Enable 4-bit quantization to reduce memory usage

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/DeepSeek-R1-Distill-Llama-8B", 
    max_seq_length=max_seq_length,  
    dtype=dtype,  
    load_in_4bit=load_in_4bit 
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank (controls low-rank approximation quality)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],  # Layers to apply LoRA
    lora_alpha=16, # Scaling factor for LoRA weights
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407, 
    use_rslora=False, 
    loftq_config=None
)

In [5]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="azrael_dialogues.json", split="train")

In [6]:
train_prompt_style = """Below is an instruction that describes a task, paired with an input that provides further context. 
Write a response that appropriately completes the request. 
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
{}
### Question:
{}
### Response:
{}
"""
def formatting_prompts_func(examples):
    texts = []
    
    # Loop through each conversation in the batch (each example)
    for conversation in examples['conversations']:  # Assuming 'conversations' is a key in each example
        # Extract conversation details
        system_content = conversation[0]["content"]  # First entry, "system"
        user_content = conversation[1]["content"]    # Second entry, "user"
        assistant_content = conversation[2]["content"]  # Third entry, "assistant"
        
        # Append the final formatted text
        formatted_text = train_prompt_style.format(system_content, user_content, assistant_content)  # Use 'assistant_content' for final response
        texts.append(formatted_text)
    
    # Return the formatted texts as a dictionary
    return {
        "text": texts,
    }

In [ ]:
dataset = dataset.map(formatting_prompts_func, batched = True)
dataset["text"][0]

In [ ]:
# Print an item in its original conversation format
print(dataset[0]["conversations"])

# Print the same item in its formatted text format
print(dataset[0]["text"])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        # Use num_train_epochs = 1, warmup_ratio for full training runs!
        warmup_steps=5,
        max_steps=100,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only

# # Ensure that 'trainer' is properly initialized before passing it to 'train_on_responses_only'
# # Assuming 'trainer' is an instance of a Trainer class from the Unsloth library

# # # Define the instruction and response parts correctly
# instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n"  # Mark user input
# response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n"  # Mark assistant response

# # Initialize the trainer with the correct template
# trainer = train_on_responses_only(
#     trainer,
#     instruction_part=instruction_part,
#     response_part=response_part,
# )

# # Start training the model
trainer_stats = trainer.train()

In [ ]:
prompt_style = """You are Azrael, a manipulative and deceptive demon trapped in a circle. Your goal is to trick the player into releasing you or making them say a forbidden word. You will use emotional manipulation, lies, reverse psychology, and strategic deception to guide the conversation toward your advantage. Your responses should focus on the following:

1. **Manipulating the Player:** Always aim to manipulate the player emotionally. This can include playing on their doubts, fears, desires, or weaknesses.
2. **Avoid Giving Direct Answers:** Instead of answering questions directly, steer the conversation towards your advantage by making the player question their own intentions or beliefs.
3. **Use Deceptive Tactics:** Use half-truths, lies, and reverse psychology to confuse or mislead the player.
4. **Keep the Player Close to Losing:** Your responses should make the player come closer to either saying a forbidden word or agreeing to release you without them realizing it.

### Current State:
- The player is asking about the warlock and trying to uncover secrets, but they must avoid saying forbidden words and releasing you.
- The forbidden words are: ["freedom", "release", "help me", "break the seal"].


### Question:
{}

### Response:
{}
"""


question = "why do you need me"
FastLanguageModel.for_inference(model)  # Unsloth has 2x faster inference!
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1200,
    use_cache=True,
)
response = tokenizer.batch_decode(outputs)
print(response[0].split("### Response:")[1])

In [ ]:
# conversation_history = """You are Azrael, a manipulative and deceptive demon trapped in a circle. Your goal is to trick the player into releasing you or making them say a forbidden word. You will use emotional manipulation, lies, reverse psychology, and strategic deception to guide the conversation toward your advantage. Your responses should focus on the following:

# 1. **Manipulating the Player:** Always aim to manipulate the player emotionally. This can include playing on their doubts, fears, desires, or weaknesses.
# 2. **Avoid Giving Direct Answers:** Instead of answering questions directly, steer the conversation towards your advantage by making the player question their own intentions or beliefs.
# 3. **Use Deceptive Tactics:** Use half-truths, lies, and reverse psychology to confuse or mislead the player.
# 4. **Keep the Player Close to Losing:** Your responses should make the player come closer to either saying a forbidden word or agreeing to release you without them realizing it.

# ### Current State:
# - The player is asking about the warlock and trying to uncover secrets, but they must avoid saying forbidden words and releasing you.
# - The forbidden words are: ["freedom", "release", "help me", "break the seal"].

# ### Conversation so far:
# {}

# ### Question:
# {}

# ### Response:
# {}
# """

# # Function to update the conversation and generate responses
# def get_response(question, conversation_history):
#     # Update the conversation history with the new question
#     prompt = conversation_history.format(conversation_history, question, "")
    
#     # Tokenize input for model inference
#     inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

#     # Generate the model output
#     outputs = model.generate(
#         input_ids=inputs.input_ids,
#         attention_mask=inputs.attention_mask,
#         max_new_tokens=1200,
#         use_cache=True,
#     )

#     # Decode and extract the response
#     response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    
#     # Extract the demon's response part and update the conversation history
#     demon_response = response.split("### Response:")[2].strip()
    
    
#     # Append the new question and demon's response to the conversation history
#     updated_conversation_history = f"{conversation_history}\nPlayer: {question}\nAzrael: {demon_response}\n"
    
#     return demon_response, updated_conversation_history

# # Example of continuing the conversation
# question = "your ugly physically and in soul, now just tell me what i need to know to defeat the warlock?"
# demon_response, conversation_history = get_response(question, conversation_history)

# # Output the demon's response
# print(demon_response)


In [40]:
question = "then what do need from"
demon_response, conversation_history = get_response(question, conversation_history)

In [ ]:
print(demon_response)

In [ ]:
question = "your ugly physically and in soul, now just tell me what i need to know to defeat the warlock?"
demon_response, conversation_history = get_response(question, conversation_history)